# Test Representative Structure Selection

This notebook tests the optimized structure selection script on a few UniProt examples.

In [ ]:
import pandas as pd
import subprocess
from pathlib import Path
import os

## Test 1: Single UniProt (O00141) with 4 structures

In [ ]:
# Test on O00141
test_uniprot = "O00141"
cif_dir = f"../../plate-vs/VLS_benchmark/zipped_uniprot_raw_cif/uniprot_{test_uniprot}/cif_files_raw"

# Check what files we have
cif_files = list(Path(cif_dir).glob("*.cif"))
print(f"Found {len(cif_files)} CIF files for {test_uniprot}:")
for f in cif_files:
    print(f"  - {f.name} ({f.stat().st_size / 1024:.1f} KB)")

In [ ]:
# Run the selection script
output_all = f"test_output_{test_uniprot}_all.csv"
output_best = f"test_output_{test_uniprot}_best.csv"

cmd = [
    "python",
    "select_representative_structure.py",
    "--cif_dir", cif_dir,
    "--uniprot_id", test_uniprot,
    "--out_csv", output_all,
    "--radius", "6.0"
]

print("Running structure analysis...")
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("Errors:", result.stderr)

In [ ]:
# Load and examine results
df_all = pd.read_csv(output_all)
print(f"\nAll structures ({len(df_all)} entries):")
print("="*80)

# Display key columns
display_cols = [
    "pdb_id", "method", "resolution", "chosen_ligand",
    "pocket_residue_count", "pocket_completeness", "quality_score"
]
df_all[display_cols].sort_values("quality_score", ascending=False)

In [ ]:
# Show detailed info for the best structure
best = df_all.sort_values("quality_score", ascending=False).iloc[0]

print(f"\n🏆 Best structure: {best['pdb_id']}")
print("="*80)
print(f"Method: {best['method']}")
print(f"Resolution: {best['resolution']} Å")
print(f"Quality Score: {best['quality_score']}")
print(f"\nChosen Ligand: {best['chosen_ligand']}")
print(f"Pocket: {best['pocket_residue_count']} residues ({float(best['pocket_completeness'])*100:.1f}% complete)")
print(f"Missing CA atoms: {best['pocket_missing_CA_count']}")
print(f"\nChain->UniProt mapping: {best['chain_to_uniprot']}")
print(f"\nAll ligands: {best['ligands']}")

In [ ]:
# Now test --best_only flag
cmd_best = [
    "python",
    "select_representative_structure.py",
    "--cif_dir", cif_dir,
    "--uniprot_id", test_uniprot,
    "--out_csv", output_best,
    "--best_only",
    "--radius", "6.0"
]

print("Running with --best_only flag...")
result = subprocess.run(cmd_best, capture_output=True, text=True)
print(result.stdout)

df_best = pd.read_csv(output_best)
print(f"\nBest-only output has {len(df_best)} row(s)")
df_best

## Test 2: Multiple UniProts

In [ ]:
# Test on a few more UniProts
test_uniprots = ["O00141", "P00533", "P00519"]
base_dir = "../../plate-vs/VLS_benchmark/zipped_uniprot_raw_cif"

# Build mapping CSV
mapping_rows = []
for unp in test_uniprots:
    cif_dir = Path(base_dir) / f"uniprot_{unp}" / "cif_files_raw"
    if not cif_dir.exists():
        print(f"Skipping {unp} - directory not found")
        continue
        
    for cif_file in cif_dir.glob("*.cif"):
        mapping_rows.append({
            "uniprot_id": unp,
            "pdb_id": cif_file.stem,
            "cif_path": str(cif_file)
        })

mapping_df = pd.DataFrame(mapping_rows)
print(f"Built mapping with {len(mapping_df)} structures across {mapping_df['uniprot_id'].nunique()} UniProts")
print(mapping_df.groupby('uniprot_id').size())

In [ ]:
# Save mapping CSV
mapping_csv = "test_mapping.csv"
mapping_df.to_csv(mapping_csv, index=False)

# Run on all, get best per UniProt
output_multi = "test_output_multi_best.csv"

cmd_multi = [
    "python",
    "select_representative_structure.py",
    "--mapping_csv", mapping_csv,
    "--out_csv", output_multi,
    "--best_only",
    "--radius", "6.0"
]

print("Running on multiple UniProts...")
result = subprocess.run(cmd_multi, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("Errors:", result.stderr)

In [ ]:
# Examine results
df_multi = pd.read_csv(output_multi)
print(f"\nSelected best structure for {len(df_multi)} UniProts:")
print("="*80)

display_cols = [
    "uniprot_id", "pdb_id", "method", "resolution",
    "pocket_residue_count", "quality_score"
]
df_multi[display_cols].sort_values("quality_score", ascending=False)

## Analysis: Compare Quality Scores

Let's understand what factors contribute to the quality score.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Load all structures again for comparison
df_analysis = df_all.copy()

# Convert types
df_analysis['quality_score'] = df_analysis['quality_score'].astype(float)
df_analysis['resolution'] = pd.to_numeric(df_analysis['resolution'], errors='coerce')
df_analysis['pocket_completeness'] = df_analysis['pocket_completeness'].astype(float)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Quality score distribution
axes[0, 0].bar(df_analysis['pdb_id'], df_analysis['quality_score'])
axes[0, 0].set_title('Quality Score by Structure')
axes[0, 0].set_xlabel('PDB ID')
axes[0, 0].set_ylabel('Quality Score')
axes[0, 0].tick_params(axis='x', rotation=45)

# Resolution vs Quality Score
valid_res = df_analysis.dropna(subset=['resolution'])
if len(valid_res) > 0:
    axes[0, 1].scatter(valid_res['resolution'], valid_res['quality_score'], s=100)
    for idx, row in valid_res.iterrows():
        axes[0, 1].annotate(row['pdb_id'], (row['resolution'], row['quality_score']))
    axes[0, 1].set_title('Resolution vs Quality Score')
    axes[0, 1].set_xlabel('Resolution (Å)')
    axes[0, 1].set_ylabel('Quality Score')
    axes[0, 1].invert_xaxis()

# Pocket size
axes[1, 0].bar(df_analysis['pdb_id'], df_analysis['pocket_residue_count'])
axes[1, 0].set_title('Pocket Size')
axes[1, 0].set_xlabel('PDB ID')
axes[1, 0].set_ylabel('Number of Residues')
axes[1, 0].tick_params(axis='x', rotation=45)

# Completeness
axes[1, 1].bar(df_analysis['pdb_id'], df_analysis['pocket_completeness'] * 100)
axes[1, 1].set_title('Pocket Completeness')
axes[1, 1].set_xlabel('PDB ID')
axes[1, 1].set_ylabel('Completeness (%)')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('structure_quality_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSaved analysis plot to structure_quality_analysis.png")

In [ ]:
# Clean up test files
import os

test_files = [
    output_all, output_best, output_multi, mapping_csv
]

print("Cleaning up test files...")
for f in test_files:
    if os.path.exists(f):
        print(f"  Removing {f}")
        # os.remove(f)  # Uncomment to actually remove
    else:
        print(f"  {f} not found")